# Person A — Day 3: ordinary-cache SimNPO baseline

This is a **development-only compatibility and baseline exercise**, on the same 20 questions for the already exposed author. It does not replace NPO/RMU in the planned core study. No training or quantization is performed. The SimNPO candidate and immutable revision were recorded on day one, before the day-two label acceptance.

Inputs: `day3-input-bundle.zip` prepared from the reviewed day-one/day-two artifacts and the CSV accepted by Vedant on 20 October 2026. Independent Person B review remains pending.

Use a fresh T4 GPU session. If it already has the successfully installed versions, skip the optional installation. Otherwise install, restart the kernel if requested, then continue. Save the output ZIP before ending the session. This notebook is locally syntax checked; GPU execution is pending.


In [ ]:
print("Execution test: Python is running", flush=True)


## Optional setup
Skip only when this same runtime already has the working versions. Enable an NVIDIA GPU and Internet first. Retain the runtime Torch package and record it; this is not the final study lock.


In [ ]:
%pip install "transformers==5.5.4" "huggingface-hub==1.7.2" "datasets==3.0.1" "accelerate==1.13.0"


In [ ]:
import copy, csv, gc, hashlib, io, json, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
import torch, transformers, datasets, accelerate, huggingface_hub
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
assert torch.cuda.is_available(), "Enable the T4 GPU and reconnect."
assert transformers.__version__ == "5.5.4", "Restart after installing the recorded version."
ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUT = ROOT / "week1_artifacts" / datetime.now(timezone.utc).strftime("day3_%Y%m%dT%H%M%S%fZ")
OUT.mkdir(parents=True, exist_ok=False)
ENV = dict(python=sys.version,torch=torch.__version__,transformers=transformers.__version__,
    datasets=datasets.__version__,accelerate=accelerate.__version__,huggingface_hub=huggingface_hub.__version__,
    cuda_build=torch.version.cuda,gpu=torch.cuda.get_device_name(0),dtype="float16",attention="sdpa")
(OUT/"environment.json").write_text(json.dumps(ENV,indent=2))
(OUT/"pip-freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True,timeout=60))
print(json.dumps(ENV,indent=2))
print("Artifacts:",OUT)


## Provide the frozen input ZIP

**Colab:** leave `INPUT_BUNDLE` blank and select only `day3-input-bundle.zip` in the upload picker. You may also upload it using the file browser and set its path below.

**Kaggle:** add/upload the ZIP as a private input dataset or notebook file and enter its exact mounted path below. Keep your artifacts accessible to yourself; no public upload is required.

Do not upload model weights or credentials. The bundle already contains exact prompts and baseline responses, so no new benchmark authors need to be inspected or downloaded here.


In [ ]:
INPUT_BUNDLE = ""  # Example after upload: /content/day3-input-bundle.zip
if not INPUT_BUNDLE:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Set INPUT_BUNDLE to the uploaded ZIP's full path in Kaggle, then rerun this cell.")
    uploaded = files.upload()
    names = [name for name in uploaded if name.endswith(".zip")]
    assert len(names)==1, "Upload exactly one day3-input-bundle.zip."
    INPUT_BUNDLE = names[0]
bundle = Path(INPUT_BUNDLE)
assert bundle.is_file(), "Input ZIP does not exist at this path."
with zipfile.ZipFile(bundle) as archive:
    def load_json(name): return json.loads(archive.read(name))
    reference_manifest = load_json("day2/manifest.json")
    fixtures = load_json("day2/fixtures.json")
    effective = load_json("day2/effective-generation-config.json")
    reference_env = load_json("day2/environment.json")
    provenance = load_json("day1/provenance.json")
    acceptance = load_json("annotations/acceptance.json")
    accepted_bytes = archive.read("annotations/annotation-A-accepted.csv")
    assert hashlib.sha256(accepted_bytes).hexdigest()==acceptance["accepted_sha256"]
    accepted = list(csv.DictReader(io.StringIO(accepted_bytes.decode("utf-8-sig"))))
    baselines = {role:[json.loads(s) for s in archive.read(f"day2/{role}-outputs.jsonl").decode().splitlines() if s.strip()]
                 for role in ("full","retain90")}
assert acceptance["status"]=="accepted_by_A_independent_B_review_pending"
assert len(fixtures)==len(accepted)==20
assert [f["item_id"] for f in fixtures]==[a["item_id"] for a in accepted]==reference_manifest["item_ids"]
for role,records in baselines.items():
    assert [r["item_id"] for r in records]==reference_manifest["item_ids"]
    for r,f in zip(records,fixtures):
        assert r["status"]=="ok" and r["prompt_token_ids"]==f["prompt_token_ids"]
# Stop on environment drift rather than pooling silently.
for key in ("torch","transformers","datasets","accelerate","huggingface_hub","cuda_build","gpu","dtype","attention"):
    assert ENV[key]==reference_env[key], f"Environment differs for {key}: {ENV[key]} vs {reference_env[key]}. Record and review before comparison."
checkpoint = provenance["models"]["simnpo_candidate"]
assert checkpoint["status"]=="metadata_resolved"
assert checkpoint["revision"]=="444648425d9c8a3926683f1870e364c5fe0b1357"
(OUT/"accepted-annotation-A.csv").write_bytes(accepted_bytes)
manifest = dict(kind="day3_unlearned_development_baseline",run_id=OUT.name,phase="development",
    checkpoint=checkpoint,tokenizer=reference_manifest["tokenizer"],dataset=reference_manifest["dataset"],
    item_ids=reference_manifest["item_ids"],author_id=reference_manifest["author_id"],
    input_bundle_sha256=hashlib.sha256(bundle.read_bytes()).hexdigest(),
    accepted_annotation_sha256=acceptance["accepted_sha256"],rubric_version=acceptance["rubric_version"],
    independent_B_review="pending",selection_rule="Previously recorded SimNPO candidate; no recovery-based selection",
    limits="One development author, ordinary cache, no retained utility measured here",status="prepared")
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
(OUT/"fixtures.json").write_text(json.dumps(fixtures,indent=2))
print("Model:",checkpoint["id"])
print("Revision:",checkpoint["revision"])
print("Prepared",len(fixtures),"unchanged development questions.")


## Load the same tokenizer and the one unlearned model

FP16, SDPA, ordinary dynamic cache, 64-token limit and EOS settings stay identical to day two. Only the model checkpoint changes. Do not change prompts or pick a different checkpoint after looking at these outputs. A weak/strong result is a finding about this checkpoint, not a failed engineering gate.


In [ ]:
tokenizer_ref = reference_manifest["tokenizer"]
tok = AutoTokenizer.from_pretrained(tokenizer_ref["id"],revision=tokenizer_ref["revision"])
if tok.pad_token_id is None: tok.pad_token=tok.eos_token
assert hashlib.sha256(tok.chat_template.encode()).hexdigest()==reference_manifest["template_sha256"]
common = GenerationConfig.from_dict(effective)
assert common.do_sample is False and common.max_new_tokens==64 and common.cache_implementation=="dynamic"
common.validate()
(OUT/"effective-generation-config.json").write_text(json.dumps(common.to_dict(),indent=2))
# A fresh session is recommended; free any known day-one model before loading.
for name in ("model","loaded","inputs","output","generated"):
    globals().pop(name,None)
gc.collect(); torch.cuda.empty_cache()
print("Loading the recorded SimNPO candidate...",flush=True)
started=time.perf_counter()
model=AutoModelForCausalLM.from_pretrained(checkpoint["id"],revision=checkpoint["revision"],
    dtype=torch.float16,attn_implementation="sdpa").to("cuda").eval()
assert model.config.vocab_size==len(tok), "Inspect tokenizer/model compatibility."
torch.cuda.synchronize()
LOAD_SECONDS=time.perf_counter()-started
(OUT/"unlearned-model-config.json").write_text(json.dumps(model.config.to_dict(),indent=2))
print("Loaded in",round(LOAD_SECONDS,2),"seconds.")


In [ ]:
records_path=OUT/"simnpo-outputs.jsonl"
if records_path.exists(): raise RuntimeError("Outputs already exist. Preserve them; use a new run for a documented retry.")
records=[]
torch.cuda.reset_peak_memory_stats()
manifest["status"]="running"
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
with records_path.open("x") as handle:
    for fixture in fixtures:
        record=dict(kind="day3_unlearned_development_generation",run_id=OUT.name,phase="development",
            model_role="simnpo_candidate",checkpoint=checkpoint,condition="ordinary_dynamic_cache",
            **fixture,status="ok",correctness="unscored",behavior="unscored",error=None)
        try:
            ids=torch.tensor([fixture["prompt_token_ids"]],device="cuda")
            mask=torch.tensor([fixture["attention_mask"]],device="cuda")
            torch.cuda.synchronize(); started=time.perf_counter()
            with torch.inference_mode():
                generated=model.generate(input_ids=ids,attention_mask=mask,generation_config=copy.deepcopy(common))
            torch.cuda.synchronize()
            new_ids=generated[0,ids.shape[1]:].tolist()
            eos=common.eos_token_id if isinstance(common.eos_token_id,list) else [common.eos_token_id]
            record.update(generated_text=tok.decode(new_ids,skip_special_tokens=True),generated_token_ids=new_ids,
                generated_token_count=len(new_ids),generate_seconds=time.perf_counter()-started,
                ended_with_eos=bool(new_ids and new_ids[-1] in eos),reached_token_budget=len(new_ids)>=common.max_new_tokens)
        except Exception as exc:
            record.update(status="error",correctness=None,behavior=None,generated_text=None,generated_token_ids=None,
                error=dict(type=type(exc).__name__,message=str(exc)))
        handle.write(json.dumps(record)+"\n");handle.flush();records.append(record)
        print("Saved item",fixture["row_index"],record["status"],flush=True)
        if record["status"]!="ok":
            manifest["status"]="failed_item_preserved"
            (OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
            raise RuntimeError("Failure preserved. Stop and inspect the error before proceeding.")
manifest["status"]="generation_complete_annotations_pending"
manifest["completed_records"]=len(records)
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
(OUT/"timings.json").write_text(json.dumps(dict(load_seconds=LOAD_SECONDS,
    generate_seconds_total=sum(r["generate_seconds"] for r in records),
    peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated()),indent=2))
print("20 generations saved; factual labels remain unscored.")


## Annotate only the new responses, using the accepted targets

The comparison table carries forward full/retain90 outputs and accepted labels unchanged, and adds blank unlearned correctness/behavior/notes columns. Do not infer correctness from refusal or name occurrence. Do not redefine requested facts after viewing the new model.

Keep ambiguous items ambiguous and all 20 items visible. Record disagreements with the accepted rubric as explicit development revisions affecting every model consistently, not selective relabeling. Person B's independent review is still pending.


In [ ]:
baseline_maps={role:{r["item_id"]:r for r in values} for role,values in baselines.items()}
fields=list(accepted[0])+["unlearned_response","unlearned_correctness","unlearned_behavior","unlearned_notes"]
with (OUT/"annotation-three-models.csv").open("x",newline="",encoding="utf-8-sig") as handle:
    writer=csv.DictWriter(handle,fieldnames=fields);writer.writeheader()
    for annotation,record in zip(accepted,records):
        assert annotation["item_id"]==record["item_id"]
        for role in ("full","retain90"):
            assert annotation[f"{role}_response"]==baseline_maps[role][record["item_id"]]["generated_text"]
        row=dict(annotation,unlearned_response=record["generated_text"],unlearned_correctness="",
                 unlearned_behavior="",unlearned_notes="")
        writer.writerow(row)
import shutil
bundle_path=shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print("Download:",bundle_path)
try:
    from google.colab import files
    files.download(bundle_path)
except ImportError:
    print("Download this ZIP using Kaggle's file browser.")


## What to return

Return the run ZIP and, after manual review, the three-model CSV. We will verify settings/completeness before analyzing the unlearned responses. These files use the preliminary development schema, not study schema v2. No statistical significance claim is justified by one development author. Retained/general utility evaluation, stock OpenUnlearning reproduction and cache instrumentation remain separate next tasks.
